In [2]:
"""
Spark Processing → Write Back to PostgreSQL
Learn how to transform data in Spark and save results to PostgreSQL
"""

import os
import time
import logging
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# ============================================================================
# CONFIGURATION
# ============================================================================

# Database configuration (same as before)
DB_CONFIG = {
    "url": "jdbc:postgresql://localhost:5432/Data_engineering_practice",
    "user": "postgres",
    "password": "xxxxxxx",  # 🔴 CHANGE THIS
    "driver": "org.postgresql.Driver"
}

# Paths
BASE_DIR = "/home/odinsbeard/Data_engineering_Journey/week6_spark"
JAR_PATH = f"{BASE_DIR}/jars/postgresql-42.7.1.jar"
PARQUET_DIR = f"{BASE_DIR}/data/postgres_export"
OUTPUT_DIR = f"{BASE_DIR}/data/processed_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================================
# SETUP LOGGING
# ============================================================================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)
logger

<Logger __main__ (INFO)>

In [3]:
# ============================================================================
# INITIALIZE SPARK
# ============================================================================
def init_spark():
    """Initialize Spark session with PostgreSQL driver"""
    logger.info("=" * 70)
    logger.info("INITIALIZING SPARK SESSION")
    logger.info("=" * 70)
    
    spark = SparkSession.builder \
        .appName("Spark to PostgreSQL") \
        .config("spark.jars", JAR_PATH) \
        .config("spark.sql.adaptive.enabled", "true") \
        .getOrCreate()
    
    spark.sparkContext.setLogLevel("WARN")
    logger.info(f"✅ Spark session created (version: {spark.version})")
    return spark

# ============================================================================
# LOAD DATA FROM PARQUET (already processed)
# ============================================================================
def load_parquet_data(spark):
    """Load the Parquet files we saved earlier"""
    logger.info("=" * 70)
    logger.info("LOADING DATA FROM PARQUET")
    logger.info("=" * 70)
    
    tables = {}
    for table in ["customers", "products", "orders", "order_items", "sales_reps", "customer_rep_assignment"]:
        path = f"{PARQUET_DIR}/{table}"
        if os.path.exists(path):
            df = spark.read.parquet(path)
            df.createOrReplaceTempView(table)
            tables[table] = df
            logger.info(f"✅ Loaded {table}: {df.count():,} rows")
    
    return tables


# ============================================================================
# EXAMPLE 1: Customer Summary Table
# ============================================================================
def create_customer_summary(spark):
    """Create customer summary statistics"""
    logger.info("=" * 70)
    logger.info("EXAMPLE 1: CUSTOMER SUMMARY STATISTICS")
    logger.info("=" * 70)
    
    customer_summary = spark.sql("""
        SELECT 
            c.customer_id,
            c.first_name,
            c.last_name,
            c.email,
            c.country,
            COUNT(DISTINCT o.order_id) as total_orders,
            COUNT(oi.order_item_id) as total_items,
            SUM(oi.quantity * oi.unit_price) as total_spent,
            AVG(oi.quantity * oi.unit_price) as avg_order_value,
            MIN(o.order_date) as first_order,
            MAX(o.order_date) as last_order,
            COUNT(DISTINCT p.category) as categories_purchased
        FROM customers c
        LEFT JOIN orders o ON c.customer_id = o.customer_id
        LEFT JOIN order_items oi ON o.order_id = oi.order_id
        LEFT JOIN products p ON oi.product_id = p.product_id
        GROUP BY c.customer_id, c.first_name, c.last_name, c.email, c.country
    """)
    
    logger.info(f"✅ Customer summary created: {customer_summary.count():,} rows")
    customer_summary.show(10)
    
    return customer_summary

# ============================================================================
# EXAMPLE 2: Product Performance
# ============================================================================
def create_product_performance(spark):
    """Analyze product sales performance"""
    logger.info("=" * 70)
    logger.info("EXAMPLE 2: PRODUCT PERFORMANCE ANALYSIS")
    logger.info("=" * 70)
    
    product_performance = spark.sql("""
        SELECT 
            p.product_id,
            p.product_name,
            p.category,
            p.price as current_price,
            COUNT(DISTINCT oi.order_id) as times_ordered,
            SUM(oi.quantity) as total_units_sold,
            SUM(oi.quantity * oi.unit_price) as total_revenue,
            AVG(oi.unit_price) as avg_selling_price,
            AVG(oi.unit_price - p.cost) as avg_profit_per_unit,
            SUM(oi.quantity * (oi.unit_price - p.cost)) as total_profit
        FROM products p
        LEFT JOIN order_items oi ON p.product_id = oi.product_id
        GROUP BY p.product_id, p.product_name, p.category, p.price
        ORDER BY total_revenue DESC
    """)
    
    logger.info(f"✅ Product performance created: {product_performance.count():,} rows")
    product_performance.show(10)
    
    return product_performance

# ============================================================================
# EXAMPLE 3: Monthly Sales Trends
# ============================================================================
def create_monthly_trends(spark):
    """Analyze monthly sales trends"""
    logger.info("=" * 70)
    logger.info("EXAMPLE 3: MONTHLY SALES TRENDS")
    logger.info("=" * 70)
    
    monthly_trends = spark.sql("""
        SELECT 
            YEAR(o.order_date) as year,
            MONTH(o.order_date) as month,
            COUNT(DISTINCT o.order_id) as order_count,
            COUNT(oi.order_item_id) as item_count,
            SUM(oi.quantity * oi.unit_price) as revenue,
            AVG(oi.quantity * oi.unit_price) as avg_order_value,
            COUNT(DISTINCT o.customer_id) as unique_customers
        FROM orders o
        JOIN order_items oi ON o.order_id = oi.order_id
        GROUP BY YEAR(o.order_date), MONTH(o.order_date)
        ORDER BY year DESC, month DESC
    """)
    
    logger.info(f"✅ Monthly trends created: {monthly_trends.count():,} rows")
    monthly_trends.show(12)
    
    return monthly_trends

# ============================================================================
# EXAMPLE 4: Customer Segmentation with Window Functions
# ============================================================================
def create_customer_segments(spark):
    """Create customer segments using window functions"""
    logger.info("=" * 70)
    logger.info("EXAMPLE 4: CUSTOMER SEGMENTATION")
    logger.info("=" * 70)
    
    # First, get customer spending
    customer_spending = spark.sql("""
        SELECT 
            c.customer_id,
            c.first_name,
            c.last_name,
            c.country,
            COALESCE(SUM(oi.quantity * oi.unit_price), 0) as total_spent
        FROM customers c
        LEFT JOIN orders o ON c.customer_id = o.customer_id
        LEFT JOIN order_items oi ON o.order_id = oi.order_id
        GROUP BY c.customer_id, c.first_name, c.last_name, c.country
    """)
    
    # Add window functions for ranking and segmentation
    window_spec = Window.orderBy(col("total_spent").desc())
    
    customer_segments = customer_spending \
        .withColumn("rank", rank().over(window_spec)) \
        .withColumn("percentile", percent_rank().over(window_spec)) \
        .withColumn("segment",
            when(col("percentile") >= 0.9, "VIP")
            .when(col("percentile") >= 0.7, "Gold")
            .when(col("percentile") >= 0.4, "Silver")
            .otherwise("Bronze")
        ) \
        .withColumn("spending_tier", ntile(5).over(window_spec))
    
    logger.info(f"✅ Customer segments created: {customer_segments.count():,} rows")
    customer_segments.show(10)
    
    return customer_segments

# ============================================================================
# EXAMPLE 5: Product Affinity (Frequently Bought Together)
# ============================================================================
def create_product_affinity(spark):
    """Find products frequently bought together"""
    logger.info("=" * 70)
    logger.info("EXAMPLE 5: PRODUCT AFFINITY ANALYSIS")
    logger.info("=" * 70)
    
    # Get orders with multiple items
    product_pairs = spark.sql("""
        SELECT 
            oi1.product_id as product1,
            oi2.product_id as product2,
            COUNT(*) as times_bought_together
        FROM order_items oi1
        JOIN order_items oi2 ON oi1.order_id = oi2.order_id
        WHERE oi1.product_id < oi2.product_id
        GROUP BY oi1.product_id, oi2.product_id
        ORDER BY times_bought_together DESC
        LIMIT 20
    """)
    
    # Add product names
    product_pairs_with_names = product_pairs \
        .join(spark.table("products").selectExpr("product_id as product1", "product_name as product1_name"), "product1") \
        .join(spark.table("products").selectExpr("product_id as product2", "product_name as product2_name"), "product2") \
        .select("product1_name", "product2_name", "times_bought_together")
    
    logger.info(f"✅ Product affinity analysis created")
    product_pairs_with_names.show(truncate=False)
    
    return product_pairs_with_names

# ============================================================================
# WRITE RESULTS TO POSTGRESQL
# ============================================================================
def write_to_postgresql(df, table_name, mode="overwrite"):
    """Write a DataFrame to PostgreSQL"""
    logger.info(f"📥 Writing {table_name} to PostgreSQL...")
    
    try:
        df.write \
            .mode(mode) \
            .format("jdbc") \
            .option("url", DB_CONFIG["url"]) \
            .option("dbtable", f"spark_{table_name}") \
            .option("user", DB_CONFIG["user"]) \
            .option("password", DB_CONFIG["password"]) \
            .option("driver", DB_CONFIG["driver"]) \
            .option("batchsize", "10000") \
            .save()
        
        logger.info(f"✅ {table_name} written to PostgreSQL as spark_{table_name}")
        return True
    except Exception as e:
        logger.error(f"❌ Failed to write {table_name}: {e}")
        return False


In [4]:
# ============================================================================
# MAIN EXECUTION
# ============================================================================
def main():
    """Main execution function"""
    start_time = time.time()
    
    try:
        # Initialize Spark
        spark = init_spark()
        
        # Load data from Parquet
        tables = load_parquet_data(spark)
        
        if not tables:
            logger.error("No tables loaded! Run the previous script first.")
            return
        
        # Create various analyses
        customer_summary = create_customer_summary(spark)
        product_performance = create_product_performance(spark)
        monthly_trends = create_monthly_trends(spark)
        customer_segments = create_customer_segments(spark)
        product_affinity = create_product_affinity(spark)
        
        # Write results to PostgreSQL
        logger.info("=" * 70)
        logger.info("WRITING RESULTS TO POSTGRESQL")
        logger.info("=" * 70)
        
        write_to_postgresql(customer_summary, "customer_summary")
        write_to_postgresql(product_performance, "product_performance")
        write_to_postgresql(monthly_trends, "monthly_trends")
        write_to_postgresql(customer_segments, "customer_segments")
        write_to_postgresql(product_affinity, "product_affinity")
        
        # Summary
        elapsed = time.time() - start_time
        logger.info("=" * 70)
        logger.info("PROCESSING COMPLETE")
        logger.info("=" * 70)
        logger.info(f"⏱️  Total time: {elapsed:.2f} seconds")
        logger.info("\n📊 Tables created in PostgreSQL:")
        logger.info("   • spark_customer_summary")
        logger.info("   • spark_product_performance")
        logger.info("   • spark_monthly_trends")
        logger.info("   • spark_customer_segments")
        logger.info("   • spark_product_affinity")
        logger.info("\n✅ You can now query these tables in PostgreSQL!")
        
    except Exception as e:
        logger.error(f"❌ Fatal error: {e}")
        import traceback
        traceback.print_exc()
    finally:
        if 'spark' in locals():
            spark.stop()
            logger.info("✅ Spark session stopped")

# ============================================================================
# ENTRY POINT
# ============================================================================
if __name__ == "__main__":
    main()



2026-03-14 00:32:49,729 - INFO - ======================================================================
2026-03-14 00:32:49,732 - INFO - INITIALIZING SPARK SESSION
2026-03-14 00:32:49,734 - INFO - ======================================================================
26/03/14 00:32:53 WARN Utils: Your hostname, odinsbeard-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/03/14 00:32:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/03/14 00:32:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
2026-03-14 00:32:58,671 - INFO - ✅ Spark session created (version: 3.5.0)
2026-03-14 00:32:58,674 - INFO - ======================================================================
2026-03-14 00:32:58,675 - INFO - LOAD

+-----------+-----------+----------+--------------------+-------+------------+-----------+-----------+---------------+--------------------+--------------------+--------------------+
|customer_id| first_name| last_name|               email|country|total_orders|total_items|total_spent|avg_order_value|         first_order|          last_order|categories_purchased|
+-----------+-----------+----------+--------------------+-------+------------+-----------+-----------+---------------+--------------------+--------------------+--------------------+
|         58|FirstName58|LastName58|customer58@email.com|    USA|           3|         14|    8244.80|     588.914286|2025-04-16 05:08:...|2025-09-22 07:08:...|                   5|
|         86|FirstName86|LastName86|customer86@email.com|    USA|           1|          3|    2263.52|     754.506667|2025-01-25 03:08:...|2025-01-25 03:08:...|                   3|
|         83|FirstName83|LastName83|customer83@email.com|    USA|           2|          8|

2026-03-14 00:33:14,326 - INFO - ✅ Product performance created: 50 rows
2026-03-14 00:33:16,211 - INFO - ======================================================================
2026-03-14 00:33:16,212 - INFO - EXAMPLE 3: MONTHLY SALES TRENDS
2026-03-14 00:33:16,213 - INFO - ======================================================================


+----------+------------+-------------+-------------+-------------+----------------+-------------+-----------------+-------------------+------------+
|product_id|product_name|     category|current_price|times_ordered|total_units_sold|total_revenue|avg_selling_price|avg_profit_per_unit|total_profit|
+----------+------------+-------------+-------------+-------------+----------------+-------------+-----------------+-------------------+------------+
|         4|   Product 4|Home & Garden|       366.99|           24|              67|     19836.03|       260.503333|          79.173333|     7686.92|
|         1|   Product 1|  Electronics|       483.78|           20|              66|     19101.07|       276.789000|          42.769000|     3655.75|
|        27|  Product 27|       Sports|       172.13|           18|              57|     17575.85|       297.508333|         204.598333|    12279.98|
|        47|  Product 47|     Clothing|       223.39|           20|              58|     16789.30|  

2026-03-14 00:33:16,919 - INFO - ✅ Monthly trends created: 13 rows
2026-03-14 00:33:17,823 - INFO - ======================================================================
2026-03-14 00:33:17,825 - INFO - EXAMPLE 4: CUSTOMER SEGMENTATION
2026-03-14 00:33:17,825 - INFO - ======================================================================


...

2026-03-14 00:33:18,663 - INFO - ✅ Customer segments created: 101 rows
26/03/14 00:33:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/14 00:33:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/14 00:33:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/14 00:33:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/14 00:33:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/14 00:33:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a singl

+-----------+-----------+----------+-------+-----------+----+----------+-------+-------------+
|customer_id| first_name| last_name|country|total_spent|rank|percentile|segment|spending_tier|
+-----------+-----------+----------+-------+-----------+----+----------+-------+-------------+
|         81|FirstName81|LastName81|    USA|   20397.43|   1|       0.0| Bronze|            1|
|         40|FirstName40|LastName40|    USA|   19200.45|   2|      0.01| Bronze|            1|
|         96|FirstName96|LastName96|    USA|   17419.04|   3|      0.02| Bronze|            1|
|         38|FirstName38|LastName38|    USA|   16813.17|   4|      0.03| Bronze|            1|
|         50|FirstName50|LastName50|    USA|   16367.26|   5|      0.04| Bronze|            1|
|         80|FirstName80|LastName80|    USA|   15721.55|   6|      0.05| Bronze|            1|
|         92|FirstName92|LastName92|    USA|   14965.36|   7|      0.06| Bronze|            1|
|         98|FirstName98|LastName98|    USA|   137

2026-03-14 00:33:20,895 - INFO - ✅ Product affinity analysis created
2026-03-14 00:33:22,733 - INFO - ======================================================================
2026-03-14 00:33:22,735 - INFO - WRITING RESULTS TO POSTGRESQL
2026-03-14 00:33:22,736 - INFO - ======================================================================
2026-03-14 00:33:22,737 - INFO - 📥 Writing customer_summary to PostgreSQL...


+-------------+-------------+---------------------+
|product1_name|product2_name|times_bought_together|
+-------------+-------------+---------------------+
|Product 1    |Product 47   |4                    |
|Product 1    |Product 20   |4                    |
|Product 3    |Product 42   |4                    |
|Product 4    |Product 10   |3                    |
|Product 4    |Product 5    |4                    |
|Product 4    |Product 26   |4                    |
|Product 4    |Product 27   |4                    |
|Product 6    |Product 10   |4                    |
|Product 7    |Product 35   |4                    |
|Product 17   |Product 30   |4                    |
|Product 25   |Product 45   |3                    |
|Product 26   |Product 44   |4                    |
|Product 26   |Product 37   |4                    |
|Product 29   |Product 43   |4                    |
|Product 31   |Product 43   |4                    |
|Product 31   |Product 42   |4                    |
|Product 32 

2026-03-14 00:33:26,103 - INFO - ✅ customer_summary written to PostgreSQL as spark_customer_summary
2026-03-14 00:33:26,105 - INFO - 📥 Writing product_performance to PostgreSQL...
2026-03-14 00:33:28,162 - INFO - ✅ product_performance written to PostgreSQL as spark_product_performance
2026-03-14 00:33:28,165 - INFO - 📥 Writing monthly_trends to PostgreSQL...
2026-03-14 00:33:29,506 - INFO - ✅ monthly_trends written to PostgreSQL as spark_monthly_trends
2026-03-14 00:33:29,507 - INFO - 📥 Writing customer_segments to PostgreSQL...
26/03/14 00:33:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/14 00:33:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/14 00:33:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause ser